# Процесс сериализации / десериализации модели и сохранения пайплайна

In [4]:
# Библиотека для сериализации и десериализации данных
# Для сериализации больших данных используй библиотеку joblib
import pickle

import warnings

import numpy as np

from sklearn.linear_model import LinearRegression

from sklearn.datasets import load_diabetes

from sklearn.feature_selection import SelectKBest, f_regression

from sklearn.preprocessing import MinMaxScaler

from sklearn.pipeline import Pipeline

from sklearn.base import TransformerMixin, BaseEstimator

# Убираем шум от предупреждений
warnings.filterwarnings('ignore')


class MyTransformer(TransformerMixin, BaseEstimator):
    """
    Шаблон кастомного трансформера
    """

    def __init__(self):
        """
        Здесь прописывается инициализация параметров, не зависящих от данных.
        """
        pass

    def fit(self, X, y=None):
        """
        Здесь прописывается «обучение» трансформера.
        Вычисляются необходимые для работы трансформера параметры (если они нужны).
        """
        return self

    def transform(self, X):
        """
        Здесь прописываются действия с данными.
        """
        # Создаём новый столбец как произведение первых трёх
        new_column = X[:, 0] * X[:, 1] * X[:, 2]
        # Для добавления столбца в массив нужно изменить его размер на (n_rows, 1)
        new_column = new_column.reshape(X.shape[0], 1)
        # Добавляем столбец в матрицу измерений
        X = np.append(X, new_column, axis=1)
        return X


# Загружаем датасет о диабете
X, y = load_diabetes(return_X_y=True)

# Инициализируем объект класса MyTransformer (вызывается метод __init__)
custom_transformer = MyTransformer()
# Чисто формально вызываем метод fit, но у нас он ничего не делает
custom_transformer.fit(X)
# Трансформируем исходные данные (вызывается метод transform)
X_transformed = custom_transformer.transform(X)
print('Shape before transform: {}'.format(X.shape))
print('Shape after transform: {}'.format(X_transformed.shape))

# Создаем пайплайн, который включает нормализацию, отбор признаков и обучение модели
pipe = Pipeline([
    ('FeatureEngineering', MyTransformer()),
    ('Scaling', MinMaxScaler()),
    ('FeatureSelection', SelectKBest(f_regression, k=5)),
    ('Linear', LinearRegression())
])

# Обучаем пайплайн
pipe.fit(X, y)

# Сериализуем pipeline и записываем результат в файл
# Если необходимо сохранить сериализованные пайплайны в виде потока байтов,
# нужно использовать функции dumps() и loads(), а не dump() и load().
with open('./output/my_pipeline.pkl', 'wb') as output:
    pickle.dump(pipe, output)

# Десериализуем pipeline из файла
with open('./output/my_pipeline.pkl', 'rb') as pkl_file:
    loaded_pipe = pickle.load(pkl_file)

# Сравниваем предсказания исходного и восстановленного пайплайнов
print(all(pipe.predict(X) == loaded_pipe.predict(X)))

# Предсказываем значение целевой переменной после десериализации на новых данных
features = np.array(
    [[0.00538306, -0.04464164, 0.05954058, -0.05616605, 0.02457414, 0.05286081, -0.04340085, 0.05091436, -0.00421986,
      -0.03007245]])

loaded_pipe.predict(features)

Shape before transform: (442, 10)
Shape after transform: (442, 11)
True


array([173.01985747])

# Практика применения pickle

In [5]:
# Десериализуем модель из файла pkl
with open('./input_data/model.pkl', 'rb') as pkl_file:
    loaded_model = pickle.load(pkl_file)

# Смотрим тип объекта и модель в файле
print(type(loaded_model))

# Смотрим количество признаков, на которых обучалась модель
num_features = loaded_model.coef_.shape[0]
print(f"Модель обучалась на {num_features} признаках")

# Применяем обученную модель к новым данным
features_new = [1, 1, 1, 0.661212487096872]
# Преобразуем в 4D (так как 4 признака)
features_new_4d = np.array([1, 1, 1, 0.661212487096872]).reshape(1, -1)
# Применяем к набору данных
loaded_model.predict(features_new_4d)

# У модели есть два поля (атрибута) с именами a и b.
# Создаем из них словарь с такими же именами ключей и значениями
# Аналогично: ({'a': model.a, 'b': model.b})
model_attr = {
    'a': getattr(loaded_model, 'a'),
    'b': getattr(loaded_model, 'b')
}
print(model_attr)

# Сохраняем результат в файл с помощью модуля pickle
with open('./output/model_attr.pkl', 'wb') as output:
    pickle.dump(model_attr, output)


secret word: skillfactory
how is this possible? answer is here: https://youtu.be/xm-A-h9QkXg
<class 'sklearn.linear_model._base.LinearRegression'>
Модель обучалась на 4 признаках
{'a': 5, 'b': 13}


In [6]:
# Проверяем правильность решения задания через специальный проверочный скрипт
# python3 hw1_check_ol.py model_attr.pkl
# ('secret code 2:', '3c508')

# Работа с библиотекой Nyoka
Генерация файла формата PMML (Predictive Model Markup Language) для совместимости модели с другими языками программирования.
В итоговом файле содержится вся информация для того, чтобы пайплайн можно было использовать на любом языке программирования.

In [7]:
from nyoka import skl_to_pmml
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.datasets import load_diabetes

X, y = load_diabetes(return_X_y=True)
cols = load_diabetes()['feature_names']

scaler = MinMaxScaler()
pipe = Pipeline([
    ('Scaling', MinMaxScaler()),
    ('Linear', LinearRegression())
])
# Обучение пайплайна, включающего линейную модель и нормализацию признаков
pipe.fit(X, y)
# Сохраним пайплайн в формате pmml в файл pipeline.pmml
skl_to_pmml(pipeline=pipe, col_names=cols, pmml_f_name="./output/pipeline.pmml")